In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import json

from tqdm import tqdm

/project/GCRB/Hon_lab/s223695/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
json_fp = "./config.json"
with open(json_fp, 'r') as fp:
    config = json.load(fp)

anno_bed_file = annotation_file = config["input_data"]["anno_bed_file"]
annotation_file = config["user_defined_data"]["annotation_file"]

#genome_annotaion_file = "/project/GCRB/Hon_lab/s223695/Data_project/reference/hg38/genes.gtf"
genome_annotaion_file = "/project/GCRB/Hon_lab/s223695/Data_project/reference/hg38/hg38.ncbiRefSeq.gtf"

In [3]:
if os.path.exists(annotation_file):
    annotation_df = pd.read_csv(annotation_file)
else:
    bed_df = pd.read_csv(anno_bed_file,sep="\t",names=["guide_chr","guide_start","guide_end","attr","score","strand"])
    bed_df["protospacer_target"]     = bed_df["attr"].apply(lambda x: x.split(";")[0])
    bed_df["intended_target_region"] = bed_df["attr"].apply(lambda x: x.split(";")[1])
    bed_df["gene_target"]            = bed_df["attr"].apply(lambda x: x.split(";")[2])
    bed_df["source"]                 = bed_df["attr"].apply(lambda x: x.split(";")[3])
    bed_df = bed_df.drop("attr",axis=1)
    bed_df.to_csv(annotation_file,index=None)
    annotation_df = bed_df
    
annotation_df.head()

,Unnamed: 0,guide_chr,guide_start,guide_end,score,strand,protospacer_target,intended_target_region,gene_target,source,closest_gene,closest_dist,closest_gene_target,closest_dist_target
0,0,chr9,130713821,130713839,.,+,chr9:130713821-130713839(+),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,56,ABL1,56
1,1,chr9,130713809,130713827,.,+,chr9:130713809-130713827(+),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,56,ABL1,56
2,2,chr9,130713859,130713877,.,+,chr9:130713859-130713877(+),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,56,ABL1,56
3,3,chr9,130714246,130714264,.,-,chr9:130714246-130714264(-),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,56,ABL1,56
4,4,chr9,130713865,130713883,.,+,chr9:130713865-130713883(+),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,56,ABL1,56


In [4]:
genome_anno_df = pd.read_csv(genome_annotaion_file,skiprows=5,sep="\t",names=["chrom","source","feature",
                                                                              "start","end","score","strand",
                                                                              "frame","attr"
                                                                             ]) 

In [5]:
genome_anno_df.head()

,chrom,source,feature,start,end,score,strand,frame,attr
0,chrM,ncbiRefSeq.2022-10-28,exon,14747,15887,.,+,.,"gene_id ""CYTB""; transcript_id ""rna-CYTB""; exon..."
1,chrM,ncbiRefSeq.2022-10-28,CDS,14747,15887,.,+,0,"gene_id ""CYTB""; transcript_id ""rna-CYTB""; exon..."
2,chrM,ncbiRefSeq.2022-10-28,start_codon,14747,14749,.,+,0,"gene_id ""CYTB""; transcript_id ""rna-CYTB""; exon..."
3,chrM,ncbiRefSeq.2022-10-28,transcript,14674,14742,.,-,.,"gene_id ""TRNE""; transcript_id ""rna-TRNE""; gen..."
4,chrM,ncbiRefSeq.2022-10-28,exon,14674,14742,.,-,.,"gene_id ""TRNE""; transcript_id ""rna-TRNE""; exon..."


In [6]:
np.unique(genome_anno_df["feature"])

array(['3UTR', '5UTR', 'CDS', 'exon', 'start_codon', 'stop_codon',
       'transcript'], dtype=object)

In [9]:
genome_anno_df_transcript= genome_anno_df[genome_anno_df["feature"]=="transcript"]

In [10]:
def attr_to_dict(attr_str):
    return_dict = {}
    split_1 = attr_str.strip().split(';')
    
    for attr_each in split_1[:len(split_1)-1]:
        split_2 = attr_each.strip().split(' ')
        return_dict[split_2[0]] = split_2[1].strip('"')
        
    return return_dict

In [11]:
#genome_anno_df_gene.loc[:,"gene_type"] = genome_anno_df_gene["attr"].apply(lambda x:attr_to_dict(x)["gene_type"])
#genome_anno_df_gene.loc[:,"gene_name"] = genome_anno_df_gene["attr"].apply(lambda x:attr_to_dict(x)["gene_name"])

In [12]:
# genome_anno_df_transcript.loc[:,"gene_type"] = \
#             genome_anno_df_transcript["attr"].apply(lambda x:attr_to_dict(x)["gene_type"])
genome_anno_df_transcript.loc[:,"gene_name"] = \
            genome_anno_df_transcript["attr"].apply(lambda x:attr_to_dict(x)["gene_name"])

/tmp/ipykernel_28130/1093322701.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  genome_anno_df_transcript.loc[:,"gene_name"] = \


In [13]:
# genome_anno_df_transcript = genome_anno_df_transcript[genome_anno_df_transcript["gene_type"]=="protein_coding"]

In [14]:
genome_anno_df_transcript

,chrom,source,feature,start,end,score,strand,frame,attr,gene_name
3,chrM,ncbiRefSeq.2022-10-28,transcript,14674,14742,.,-,.,"gene_id ""TRNE""; transcript_id ""rna-TRNE""; gen...",TRNE
5,chrM,ncbiRefSeq.2022-10-28,transcript,14149,14673,.,-,.,"gene_id ""ND6""; transcript_id ""rna-ND6""; gene_...",ND6
10,chrM,ncbiRefSeq.2022-10-28,transcript,12337,14148,.,+,.,"gene_id ""ND5""; transcript_id ""rna-ND5""; gene_...",ND5
15,chrM,ncbiRefSeq.2022-10-28,transcript,12266,12336,.,+,.,"gene_id ""TRNL2""; transcript_id ""rna-TRNL2""; g...",TRNL2
17,chrM,ncbiRefSeq.2022-10-28,transcript,12207,12265,.,+,.,"gene_id ""TRNS2""; transcript_id ""rna-TRNS2""; g...",TRNS2
...,...,...,...,...,...,...,...,...,...,...
4886673,chr1,ncbiRefSeq.2022-10-28,transcript,30366,30503,.,+,.,"gene_id ""MIR1302-2""; transcript_id ""NR_036051....",MIR1302-2
4886675,chr1,ncbiRefSeq.2022-10-28,transcript,29774,35418,.,+,.,"gene_id ""MIR1302-2HG""; transcript_id ""XR_00706...",MIR1302-2HG
4886679,chr1,ncbiRefSeq.2022-10-28,transcript,17369,17436,.,-,.,"gene_id ""MIR6859-1""; transcript_id ""NR_106918....",MIR6859-1
4886681,chr1,ncbiRefSeq.2022-10-28,transcript,14362,29370,.,-,.,"gene_id ""WASH7P""; transcript_id ""NR_024540.1"";...",WASH7P


In [15]:
output_list = []
for row in genome_anno_df_transcript.itertuples():
    if not row.chrom.startswith("chr"):
        continue
    
    if row.strand == "+":
        start_base = row.start
    elif row.strand == "-":
        start_base = row.end
    output_list.append([row.chrom,start_base,start_base+1,row.gene_name])
genome_anno_tss = pd.DataFrame(output_list,columns=["chrom","start","end","name"])

In [16]:
genome_anno_tss.to_csv("./genes_tss.bed",sep="\t",header=None,index=None)

In [17]:
genome_anno_tss[genome_anno_tss["name"]=="TBX5"]

,chrom,start,end,name
83320,chr12,114408442,114408443,TBX5
83321,chr12,114406144,114406145,TBX5
83322,chr12,114408012,114408013,TBX5
83323,chr12,114408012,114408013,TBX5


<h4>Load sgRNA_annotation data</h4>

In [18]:
annotation_df.head()

,Unnamed: 0,guide_chr,guide_start,guide_end,score,strand,protospacer_target,intended_target_region,gene_target,source,closest_gene,closest_dist,closest_gene_target,closest_dist_target
0,0,chr9,130713821,130713839,.,+,chr9:130713821-130713839(+),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,56,ABL1,56
1,1,chr9,130713809,130713827,.,+,chr9:130713809-130713827(+),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,56,ABL1,56
2,2,chr9,130713859,130713877,.,+,chr9:130713859-130713877(+),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,56,ABL1,56
3,3,chr9,130714246,130714264,.,-,chr9:130714246-130714264(-),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,56,ABL1,56
4,4,chr9,130713865,130713883,.,+,chr9:130713865-130713883(+),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,56,ABL1,56


In [19]:
target_gene_list = [gene.split("_")[0] for gene in np.unique(annotation_df["gene_target"]) 
                    if not gene.startswith("Element")]

target_gene_list = np.unique(target_gene_list)
print(len(target_gene_list))

255


In [27]:
target_gene_list[~np.isin(target_gene_list,genome_anno_tss["name"])]

array(['ELMSAN1'], dtype='<U8')

In [28]:
target_gene_list[target_gene_list=='ELMSAN1'] = "MIDEAS"

In [29]:
genome_anno_tss_target = genome_anno_tss[genome_anno_tss["name"].isin(target_gene_list)]

In [30]:
print(len(np.unique(genome_anno_tss_target["name"])))

255


In [31]:
closest_gene_res = []
closest_gene_dist = []
closest_gene_res_target = []
closest_gene_dist_target = []

for row in tqdm(annotation_df.itertuples(),total=annotation_df.shape[0]):
    target_region_chr,target_region_tmp = row.intended_target_region.split(":")
    target_region_start,target_region_end = target_region_tmp.split("-")
    target_region_center = int((int(target_region_start)+int(target_region_end))/2)
    
    genome_anno_tss_chr = genome_anno_tss[genome_anno_tss["chrom"]==row.guide_chr]
    genome_anno_tss_target_chr = genome_anno_tss_target[genome_anno_tss_target["chrom"]==row.guide_chr]
    
    dist_array = np.abs(genome_anno_tss_chr["start"]-target_region_center).values
    dist_array_target = np.abs(genome_anno_tss_target_chr["start"]-target_region_center).values
 
    min_index = np.argmin(dist_array)
    min_index_target = np.argmin(dist_array_target)
    
    closest_gene_res.append(genome_anno_tss_chr.iat[min_index,3])
    closest_gene_dist.append(dist_array[min_index])
    closest_gene_res_target.append(genome_anno_tss_target_chr.iat[min_index_target,3])
    closest_gene_dist_target.append(dist_array_target[min_index_target])

100%|██████████| 13882/13882 [01:59<00:00, 116.33it/s]


In [32]:
annotation_df["closest_gene"] = closest_gene_res
annotation_df["closest_dist"] = closest_gene_dist
annotation_df["closest_gene_target"] = closest_gene_res_target
annotation_df["closest_dist_target"] = closest_gene_dist_target

In [33]:
annotation_df.to_csv(annotation_file)